In [1]:
import wandb
import pandas as pd
import numpy as np

import json

api = wandb.Api()

In [2]:
def linearize_dict(d, parent_key='', sep='.'):
    """
    Linearizes a nested dictionary into a flat dictionary with keys in the format `key1.key2`.

    :param d: The dictionary to linearize.
    :param parent_key: The current key being processed (used for recursion).
    :param sep: The separator between keys.
    :return: A flattened dictionary.
    """
    items = []
    
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(linearize_dict(v, new_key, sep).items())
        else:
            items.append((new_key, v))
    
    return dict(items)

def get_runs_df(runs):
    runs_list = []
    for run in runs: 
        runs_list.append({
            **run.summary._json_dict,
            **linearize_dict({k: v for k,v in run.config.items()
            if not k.startswith('_')}),
            **linearize_dict(run.metadata if run.metadata else {}),
            **{"name": run.name, "id": run.id}
            })

    runs_df = pd.DataFrame(runs_list)
    return runs_df

## IAUC %

In [3]:
iauc_p_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.percentage": {"$ne": None}},
            {"tags": {"$nin": ["Broken"]}},
            {"$or": [
                {'config.dataset.datasets.val_pascal5i_N1K1.name': "pascal"},
                {'config.dataset.datasets.val_coco20i_N1K1.name': "coco"},
            ]}
        ]
    }
)

In [4]:
iauc_p_runs_df = get_runs_df(iauc_p_runs)

In [6]:
res_df = iauc_p_runs_df.copy()

dataset_names = ["val_pascal5i_N1K1", "val_coco20i_N1K1"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1)

iauc_cols = [f"{name}_iauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."


renamings_dict = {
    "explainer.name": "Explanation Method",
    "config.metric.n_steps": "N. Steps",
    "config.metric.iauc": "IAUC",
    "config.metric.dauc": "DAUC",
    "model": "Model",
    "metric.percentage": "Percentage",
    "metric.loss": "Loss",
}
res_df = res_df.rename(columns=renamings_dict)

value_renamings_dict = {
    "integrated_gradients": "Integrated Gradients",
    "saliency": "Saliency",
    "affinity": "Affinity Explainer (ours)",
    "random": "Random",
    "pascal": "Pascal 5^i",
    "coco": "COCO 20^i",
    "dcama": "DCAMA",
    "dmtnet": "DMTNet",
}
res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["iAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["Loss"] = res_df["Loss"].fillna(False)

cols = ["Model", "Explanation Method", "Dataset", "Percentage", "Loss"]
res_df = res_df[cols + ["iAUC"]]

# Define custom sort order
dataset_order = {"COCO 20^i": 0, "Pascal 5^i": 1}
method_order = {
    'Random': 0,
    'Integrated Gradients': 1,
    'Saliency': 2,
    'Affinity Explainer': 3,
}
model_order = {
    'DCAMA': 0,
    'DMTNet': 1,
}

res_df = res_df.pivot_table(
    index=['Model', 'Explanation Method'],
    columns=['Dataset', 'Percentage', "Loss"],
    values='iAUC'
).sort_index(axis=1)

perc_map = {
    False: "IAUC@",
    True: "mIoULoss@"
}

# Flatten the MultiIndex columns
# Collapse last two levels into one
res_df.columns = pd.MultiIndex.from_tuples([
    (lvl0, f"{perc_map[lvl2]}{lvl1}") for lvl0, lvl1, lvl2 in res_df.columns
])


# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Model', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(model_order) if x.name == 'Model' else x.map(method_order)
)

iauc_p_df = (res_df*100).round(2)

iauc_p_df

COCO 20^i                  Pascal 5^i  \
                                 mIoULoss@0.01 mIoULoss@0.05 mIoULoss@0.01   
Model  Explanation Method                                                    
DCAMA  Random                            33.79         29.78         48.72   
       Integrated Gradients              32.33         24.20         44.36   
       Saliency                          33.93         24.53         50.33   
       Affinity Explainer (ours)         19.84         10.08         22.85   
DMTNet Random                            23.53         26.01         35.79   
       Integrated Gradients              20.73         21.66         33.05   
       Saliency                          20.15         17.46         33.29   
       Affinity Explainer (ours)         10.97          4.14         16.35   

                                                
                                 mIoULoss@0.05  
Model  Explanation Method                       
DCAMA  Random                            34.64  
       Integrated Gradients              29.54  
       Saliency                          33.09  
       Affinity Explainer (ours)          7.90  
DMTNet Random                            41.19  
       Integrated Gradients              38.80  
       Saliency                          35.57  
       Affinity Explainer (ours)          7.85

In [7]:
from tabulate import tabulate

# turn the two level index into two columns
final_to_latex = iauc_p_df.reset_index()

latex_tab = tabulate(final_to_latex, headers='keys', tablefmt='latex_booktabs', showindex=False)
print(latex_tab)

\begin{tabular}{llrrrr}
\toprule
 ('Model', '')   & ('Explanation Method', '')   &   ('COCO 20\^{}i', 'mIoULoss@0.01') &   ('COCO 20\^{}i', 'mIoULoss@0.05') &   ('Pascal 5\^{}i', 'mIoULoss@0.01') &   ('Pascal 5\^{}i', 'mIoULoss@0.05') \\
\midrule
 DCAMA           & Random                       &                            33.79 &                            29.78 &                             48.72 &                             34.64 \\
 DCAMA           & Integrated Gradients         &                            32.33 &                            24.2  &                             44.36 &                             29.54 \\
 DCAMA           & Saliency                     &                            33.93 &                            24.53 &                             50.33 &                             33.09 \\
 DCAMA           & Affinity Explainer (ours)    &                            19.84 &                            10.08 &                             22.85 &                   